In [1]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [2]:
train = pd.read_csv(
    "22121477/train_lending_club.csv"
)

test = pd.read_csv(
    "22121477/test_lending_club.csv"
)

print(train.shape)

print(test.shape)

display(train.head())
display(train.info())



(236846, 27)
(95019, 27)


,issue_d,sub_grade,term,home_ownership,fico_range_low,total_acc,pub_rec,revol_util,annual_inc,int_rate,dti,purpose,mort_acc,loan_amnt,application_type,installment,verification_status,pub_rec_bankruptcies,addr_state,initial_list_status,fico_range_high,revol_bal,id,open_acc,emp_length,loan_status,time_to_earliest_cr_line
0,2014-01-01,D1,60 months,MORTGAGE,660.0,18.0,0.0,86.8,40440.0,16.99,15.16,credit_card,1.0,17775.0,Individual,441.66,Verified,0.0,AR,f,664.0,17264.0,11024793,11.0,2.0,1,478656.0
1,2014-01-01,C4,60 months,MORTGAGE,740.0,26.0,0.0,103.5,59000.0,15.61,16.74,credit_card,4.0,29175.0,Individual,703.45,Verified,0.0,VT,f,744.0,6725.0,10596078,8.0,3.0,1,541728.0
2,2014-01-01,A4,36 months,RENT,700.0,47.0,0.0,11.4,40000.0,7.90,20.34,debt_consolidation,1.0,6000.0,Individual,187.75,Not Verified,0.0,TX,w,704.0,7613.0,10775616,27.0,1.0,1,657590.4
3,2014-01-01,D1,60 months,MORTGAGE,665.0,26.0,0.0,56.2,70000.0,16.99,23.15,debt_consolidation,7.0,15600.0,Individual,387.62,Verified,0.0,IN,f,669.0,14173.0,10765610,9.0,11.0,1,328838.4
4,2014-01-01,C3,36 months,MORTGAGE,700.0,15.0,1.0,67.1,120000.0,14.98,17.88,vacation,1.0,10000.0,Individual,346.56,Verified,0.0,LA,f,704.0,2549.0,10794837,8.0,2.0,1,305164.8


<class 'pandas.DataFrame'>
RangeIndex: 236846 entries, 0 to 236845
Data columns (total 27 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   issue_d                   236846 non-null  str    
 1   sub_grade                 236846 non-null  str    
 2   term                      236846 non-null  str    
 3   home_ownership            236846 non-null  str    
 4   fico_range_low            236846 non-null  float64
 5   total_acc                 236846 non-null  float64
 6   pub_rec                   236846 non-null  float64
 7   revol_util                236846 non-null  float64
 8   annual_inc                236846 non-null  float64
 9   int_rate                  236846 non-null  float64
 10  dti                       236846 non-null  float64
 11  purpose                   236846 non-null  str    
 12  mort_acc                  236846 non-null  float64
 13  loan_amnt                 236846 non-null  float64
 14 

None

In [3]:
#Evaluate Missing Values in Train
missing_values_train = train.isnull().sum()
print(missing_values_train)


issue_d                     0
sub_grade                   0
term                        0
home_ownership              0
fico_range_low              0
total_acc                   0
pub_rec                     0
revol_util                  0
annual_inc                  0
int_rate                    0
dti                         0
purpose                     0
mort_acc                    0
loan_amnt                   0
application_type            0
installment                 0
verification_status         0
pub_rec_bankruptcies        0
addr_state                  0
initial_list_status         0
fico_range_high             0
revol_bal                   0
id                          0
open_acc                    0
emp_length                  0
loan_status                 0
time_to_earliest_cr_line    0
dtype: int64


In [4]:
#Evaluate Missing Values in Test
missing_values_test = test.isnull().sum()
print(missing_values_test)


issue_d                        0
sub_grade                      0
term                           0
home_ownership               826
fico_range_low                 0
total_acc                      0
pub_rec                        0
revol_util                     0
annual_inc                     0
int_rate                       0
dti                            0
purpose                        0
mort_acc                       0
loan_amnt                      0
application_type               0
installment                    0
verification_status            0
pub_rec_bankruptcies           0
addr_state                     0
initial_list_status            0
fico_range_high                0
revol_bal                      0
id                             0
open_acc                       0
emp_length                     0
loan_status                 1412
time_to_earliest_cr_line       0
dtype: int64


In [5]:
# Evaluate Loan Status

train["loan_status"].value_counts(normalize = True)

loan_status
1    0.831591
0    0.168409
Name: proportion, dtype: float64

In [6]:
# Calculate Default Rate
default_rate = (train["loan_status"] == 0).sum() / len(train)
print(f"The default rate is {100 * default_rate:.2f}%")

The default rate is 16.84%


In [7]:
# FICO score analysis
fico_loan_statuses = train.groupby("fico_range_low")["loan_status"].mean()
fico_loan_statuses = fico_loan_statuses.sort_index()
fico_loan_statuses

fico_range_low
660.0    0.759205
665.0    0.776117
670.0    0.785433
675.0    0.792415
680.0    0.807474
685.0    0.820777
690.0    0.827195
695.0    0.839532
700.0    0.851690
705.0    0.859129
710.0    0.872226
715.0    0.877343
720.0    0.892391
725.0    0.895905
730.0    0.912588
735.0    0.915746
740.0    0.918728
745.0    0.928009
750.0    0.933093
755.0    0.932063
760.0    0.933449
765.0    0.928526
770.0    0.940397
775.0    0.943925
780.0    0.940823
785.0    0.930913
790.0    0.946407
795.0    0.942598
800.0    0.937710
805.0    0.959916
810.0    0.966408
815.0    0.977273
820.0    0.957547
825.0    0.960227
830.0    0.951456
835.0    1.000000
840.0    0.972222
845.0    0.909091
Name: loan_status, dtype: float64

In [8]:
train["dti_bin"] = pd.qcut(
    train["dti"], 
    10
)
dti_loan_status = train.groupby("dti_bin")["loan_status"].mean()
dti_loan_status = dti_loan_status.sort_index()
dti_loan_status

dti_bin
(-0.001, 7.56]     0.889989
(7.56, 10.75]      0.886368
(10.75, 13.265]    0.877663
(13.265, 15.58]    0.865834
(15.58, 17.89]     0.850148
(17.89, 20.29]     0.839576
(20.29, 23.02]     0.821423
(23.02, 26.19]     0.801543
(26.19, 30.38]     0.776514
(30.38, 999.0]     0.706604
Name: loan_status, dtype: float64

In [9]:
train["int_rate_bins"] = pd.qcut(
    train["int_rate"],
    10
)

int_rate_loan_status = train.groupby("int_rate_bins")["loan_status"].mean()
int_rate_loan_status = int_rate_loan_status.sort_index()
int_rate_loan_status

int_rate_bins
(5.319, 7.12]     0.977124
(7.12, 8.18]      0.959806
(8.18, 9.75]      0.936148
(9.75, 10.99]     0.918959
(10.99, 12.29]    0.885619
(12.29, 13.33]    0.854950
(13.33, 14.49]    0.812725
(14.49, 16.29]    0.764480
(16.29, 18.55]    0.661196
(18.55, 30.99]    0.524651
Name: loan_status, dtype: float64

In [10]:
train["fico_bins"] = pd.qcut(
    train["fico_range_low"],
    5
)
ficos_int_rates_loan_status = train.groupby(["fico_bins", "int_rate_bins"])["loan_status"].mean()
ficos_int_rates_loan_status = ficos_int_rates_loan_status.sort_index()
ficos_int_rates_loan_status

fico_bins         int_rate_bins 
(659.999, 670.0]  (5.319, 7.12]     0.941959
                  (7.12, 8.18]      0.956312
                  (8.18, 9.75]      0.931713
                  (9.75, 10.99]     0.914590
                  (10.99, 12.29]    0.872739
                  (12.29, 13.33]    0.847926
                  (13.33, 14.49]    0.799560
                  (14.49, 16.29]    0.752357
                  (16.29, 18.55]    0.655896
                  (18.55, 30.99]    0.503867
(670.0, 680.0]    (5.319, 7.12]     0.969671
                  (7.12, 8.18]      0.957757
                  (8.18, 9.75]      0.931912
                  (9.75, 10.99]     0.927339
                  (10.99, 12.29]    0.885093
                  (12.29, 13.33]    0.856187
                  (13.33, 14.49]    0.812980
                  (14.49, 16.29]    0.749535
                  (16.29, 18.55]    0.645321
                  (18.55, 30.99]    0.524718
(680.0, 695.0]    (5.319, 7.12]     0.966553
                  (7.1